In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/midtermNLP01/sample_submission.csv
/kaggle/input/competitions/midtermNLP01/train.csv
/kaggle/input/competitions/midtermNLP01/test.csv


# Import thư viện

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score




import os
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

# Constant


In [3]:

MODEL_NAME = "vinai/phobert-base"
NUM_LABELS = 3
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DATA_PATH = '/kaggle/input/competitions/midtermNLP01/train.csv'
TEST_DATA_PATH = '/kaggle/input/competitions/midtermNLP01/test.csv'
OUTPUT_DIR = "saved_models"
SEED = 42

In [4]:
set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:



class SentimentDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def train_epoch(model, loader, optimizer, scheduler, scaler):
    """Optimized with Mixed Precision (AMP)"""
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        # Mixed Precision Context
        with torch.cuda.amp.autocast():
            outputs = model(**batch)
            loss = outputs.loss
            
        # Scaling Loss
        scaler.scale(loss).backward()
        
        # Step and Unscale
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    return total_loss / len(loader)

def eval_epoch(model, loader):
    """Optimized with inference_mode and faster metrics"""
    model.eval()
    preds, gold = [], []
    
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            labels = batch["labels"].to(DEVICE)
            # Remove labels from batch to pass to model
            model_batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            
            outputs = model(**model_batch)
            logits = outputs.logits
            
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            
    # Multiclass F1 - change to 'weighted' if classes are imbalanced
    return f1_score(gold, preds, average='macro')

def predict(model, loader):
    """Prediction function that returns probabilities for ensembling"""
    model.eval()
    all_probs = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Predicting", leave=False):
            model_batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            logits = model(**model_batch).logits
            probs = torch.softmax(logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

def main():
    # 1. Load and Clean Data
    df = pd.read_csv(TRAIN_DATA_PATH)
    df = df.dropna(subset=["sentence", "sentiment"])
    df["sentiment"] = df["sentiment"].astype(int)
    
    # 2. Setup Tokenizer and Fold
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    
    # 3. GLOBAL PRE-TOKENIZATION (Optimizes speed)
    print("Tokenizing entire dataset once...")
    full_encodings = tokenizer(
        df["sentence"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    val_scores = []
    best_overall = 0.0
    best_overall_path = None
    
    # Storage for cross-validation predictions if ensembling on test set
    test_probs_total = []

    # 4. Training Loop
    labels_array = df["sentiment"].values
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, labels_array)):
        print(f"\n=== Fold {fold + 1} ===")
        
        # Optimized Slice: No redundant tokenization here
        train_encodings = {k: v[train_idx] for k, v in full_encodings.items()}
        val_encodings = {k: v[val_idx] for k, v in full_encodings.items()}
        
        train_ds = SentimentDataset(train_encodings, labels_array[train_idx])
        val_ds = SentimentDataset(val_encodings, labels_array[val_idx])

        # num_workers > 0 and pin_memory=True for faster batch transfers
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
        model.to(DEVICE)
        
        # Weight Decay usually helps
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        total_steps = len(train_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps,
        )
        
        # Mixed Precision Scaler
        scaler = torch.cuda.amp.GradScaler()

        best_val_f1 = 0
        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
            val_f1 = eval_epoch(model, val_loader)
            print(f"Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f} — val_f1: {val_f1:.4f}")
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                # Save best for this fold
                fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
                os.makedirs(fold_dir, exist_ok=True)
                model.save_pretrained(fold_dir)
                tokenizer.save_pretrained(fold_dir)

        print(f"Fold {fold+1} Finished. Best val F1: {best_val_f1:.4f}")
        val_scores.append(best_val_f1)

        if best_val_f1 > best_overall:
            best_overall = best_val_f1
            best_overall_path = os.path.join(OUTPUT_DIR, f"fold_{fold}")

    print("\n=== CV Results ===")
    print(f"Mean F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")
    print(f"Best fold path: {best_overall_path} (F1 = {best_overall:.4f})")

    # 5. TEST PREDICTION (ENSEMBLED)
    if os.path.exists(TEST_DATA_PATH):
        print("\n=== Predicting on Test Set (5-Fold Ensemble) ===")
        test_df = pd.read_csv(TEST_DATA_PATH)
        test_encodings = tokenizer(
            test_df["sentence"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        test_ds = SentimentDataset(test_encodings)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        final_probs = np.zeros((len(test_df), NUM_LABELS))
        
        # Predict with each fold model and average probabilities
        for fold in range(5):
            print(f"Predicting with model from fold {fold}...")
            fold_model_path = os.path.join(OUTPUT_DIR, f"fold_{fold}")
            fold_model = AutoModelForSequenceClassification.from_pretrained(fold_model_path).to(DEVICE)
            final_probs += predict(fold_model, test_loader)
            
        final_preds = np.argmax(final_probs, axis=1)
        
        submission = pd.DataFrame({"id": test_df["id"], "sentiment": final_preds})
        submission.to_csv("submission.csv", index=False)
        print("Submission saved to submission.csv")

if __name__ == "__main__":
    main()


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing entire dataset once...


...

Predicting:   0%|          | 0/152 [00:00<?, ?it/s]

Predicting with model from fold 4...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting:   0%|          | 0/152 [00:00<?, ?it/s]

Submission saved to submission.csv
